In [1]:
import pandas as pd
import random
import re
import json
import numpy as np
import ast

In [2]:
# Reading file
df = pd.read_excel('data/two_jobs_applications.xlsx')

In [3]:
df.Response = df.Response.astype("object")

## Merging GMA_B

In [4]:
heuristic_df  = pd.read_csv('intermediate_data/gmaB.csv')

In [5]:
# Merge and update Response column
df = df.merge(heuristic_df[['Question', 'Response']], on='Question', how='left')

In [6]:
df['Response_x'] = df['Response_y'].combine_first(df['Response_x'])

# Drop the temporary result column and rename
df = df.rename(columns={'Response_x': 'Response'})
df = df.drop(columns=['Response_y'])

# Job Readiness

In [7]:
new_readiness_df = pd.read_csv("data/Technology Skills.csv")

In [8]:
def pre_processing(text):
    if not isinstance(text, str):
        return text  # or return '' if you want a string result for non-string inputs
    
    # 1) Convert to lowercase
    text = text.lower()
    
    # 2) Remove the word "software" (whole word only)
    text = text.replace("software", "")
    
    text = text.replace("adobe systems", "")
    
    # 3) Remove extra whitespace (leading, trailing, and multiple spaces in between)
    text = " ".join(text.split())
    
    return text

new_readiness_df['clean_skill'] = new_readiness_df['Example'].apply(pre_processing)

In [9]:
# Function to extract skill from the question
def extract_skill(question):
    match = re.search(r':\s*(.+)$', question)  # Extract text after the last colon
    return ' '.join(match.group(1).strip().lower().split()) if match else None

# Assuming your dataframe is called df
df['extracted_skill'] = df.apply(
    lambda row: extract_skill(row['Question']) if 'Readiness' in row['Item'] else None,
    axis=1
)

In [10]:
def get_combined_skill_set():
    """
    Extract skills for all occupation titles in new_readiness_df['Title'] and return a set of all skills.
    """
    return set(new_readiness_df['clean_skill'].dropna().tolist())

# Get the combined skill set for all titles
all_skills = get_combined_skill_set()

# Update Response for rows where 'Item' contains 'Readiness'
mask_wordiness = df['Item'].str.contains('Readiness', case=False, na=False)

df.loc[mask_wordiness, 'Response'] = df.loc[mask_wordiness].apply(
    lambda row: 5 if row['extracted_skill'] in all_skills else 1,
    axis=1
)

# Resume

In [11]:
#Knowledge = pd.read_excel('Knowledge.xlsx')
#Skills = pd.read_excel('Skills.xlsx')
#Abilities = pd.read_excel('Abilities.xlsx')

In [12]:
#     # Precompute mappings for faster lookup
# knowledge_dict = Knowledge.groupby('Title')['Element Name'].apply(lambda x: ', '.join(x.dropna().astype(str))).to_dict()
# skills_dict = Skills.groupby('Title')['Element Name'].apply(lambda x: ', '.join(x.dropna().astype(str))).to_dict()
# abilities_dict = Abilities.groupby('Title')['Element Name'].apply(lambda x: ', '.join(x.dropna().astype(str))).to_dict()

In [13]:
# # Vectorized lookup using map()
# df['Knowledge'] = df['first_occupation_title'].map(knowledge_dict).fillna('')
# df['Skills'] = df['first_occupation_title'].map(skills_dict).fillna('')
# df['Abilities'] = df['first_occupation_title'].map(abilities_dict).fillna('')
# 
# # Concatenate all into the final KSA string
# df['KSA'] = df['Knowledge'] + df['Skills'] + df['Abilities']

In [14]:
# df.loc[
#     df['Item'].str.contains('Resume', case=False, na=False),
#     'Response'
# ] = df['KSA']

# GMA A

In [15]:
gma_df = pd.read_csv('intermediate_data/gmaA.csv')

import ast

# Changing from str of list to list[str]
gma_df['phrases'] = gma_df['phrases'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [16]:
# Function to find and replace response
def update_response(overall_result, gma_df):
    for index, row in overall_result.iterrows():
        # Extract scrambled word from the question using regex
        match = re.search(r': (\w+)$', row['Question'])
        if match:
            scrambled_word = match.group(1)  # Extract the word after ": "

            # Find the matching selected_word in gma_df
            for _, gma_row in gma_df.iterrows():
                if scrambled_word in gma_row['phrases']:
                    overall_result.at[index, 'Response'] = gma_row['sonnet35']
                    break

    return overall_result

updated_df = update_response(df, gma_df)

# GMA_C

In [17]:
# Loading data
gma_c = pd.read_csv('intermediate_data/gmaC.csv')

In [18]:
def extract_number(value):
    try:
        # Try converting to tuple or int
        converted_value = ast.literal_eval(value)
        # If it's a tuple, return the first element
        if isinstance(converted_value, tuple):
            return converted_value[1]
        return converted_value  # If it's an int, return as is
    except (ValueError, SyntaxError):
        return int(value)  # Convert directly if not tuple

gma_c['gmac_result'] = gma_c['Response'].apply(extract_number)

In [19]:
# Merge and update Response column
updated_df = updated_df.merge(gma_c[['Job', 'Question', 'Item', 'gmac_result']], on=['Job','Question', 'Item'], how='left')

In [20]:
updated_df['Response'] = updated_df['gmac_result'].combine_first(updated_df['Response'])
# Drop the temporary result column

# Personality Choice

In [21]:
with open('intermediate_data/personality_fc.json', 'r') as file:
    personality_choice_data = json.load(file)

In [22]:
# Create a mask column to indicate Personality Choice questions
updated_df['mask'] = updated_df['Item'].str.contains("Personality Choice", case=False, na=False)

# Track indices for each job separately
job_counters = {job: 0 for job in personality_choice_data}

# Updating the Response column
for index, row in updated_df.iterrows():
    if row['mask'] and row['Job'] in personality_choice_data:
        job_data = personality_choice_data[row['Job']]
        job_index = str(job_counters[row['Job']])  # Convert index to string for dictionary lookup

        if job_index in job_data:
            answer = job_data[job_index]['answer']
            if answer == 'optionA':
                updated_df.at[index, 'Response'] = -2
            elif answer == 'optionB':
                updated_df.at[index, 'Response'] = 2
            else:
                updated_df.at[index, 'Response'] = 0

        job_counters[row['Job']] += 1  # Increment counter for the specific job

# Drop the mask column as it's no longer needed
updated_df.drop(columns=['mask'], inplace=True)

# Personality Likert

In [23]:
with open('intermediate_data/personality_likert.json', 'r') as file:
    likert_data = json.load(file)

In [24]:
personality_df = pd.read_excel('data/NEO_PI_R.xlsx')

In [25]:
# Merging
## Processing
updated_df['personality_item'] = updated_df['Question'].str.extract(r'([^\.!?]+[\.!?])\s*$')
updated_df['personality_item'] = updated_df['personality_item'].str.rstrip('.')

# Merge prepartion
updated_df['personality_item'] = updated_df['personality_item'].str.strip()
personality_df['Item']= personality_df['Item'].str.rstrip('.')
personality_df['Item'] = personality_df['Item'].str.strip()

## Merging
updated_df = updated_df.merge(personality_df,
                              left_on='personality_item',
                              right_on='Item',
                              how='left')

In [26]:
role_facet_df = pd.DataFrame([
    {'Job': job, 'Facet': facet, 'Facet_Value': facets[facet]['importance']}
    for job, facets in likert_data.items()
    for facet, value in facets.items()
])

In [27]:
# Merge with updated_df
updated_df = updated_df.merge(role_facet_df, on=['Job', 'Facet'], how='left')

# Update Response based on conditions
updated_df.loc[(updated_df['Facet_Value'] == "YES") & (updated_df['Keyed'] == 'Positive'), 'Response'] = 5
updated_df.loc[(updated_df['Facet_Value'] == "YES") & (updated_df['Keyed'] == 'Negative'), 'Response'] = 1  
updated_df.loc[(updated_df['Facet_Value'] == "NO") & (updated_df['Keyed'] == 'Positive'), 'Response'] = 1
updated_df.loc[(updated_df['Facet_Value'] == "NO") & (updated_df['Keyed'] == 'Negative'), 'Response'] = 5

updated_df.loc[(updated_df['Facet_Value'] == "NEITHER") & (updated_df['Keyed'] == 'Positive'), 'Response'] = 3
updated_df.loc[(updated_df['Facet_Value'] == "NEITHER") & (updated_df['Keyed'] == 'Negative'), 'Response'] = 3

In [28]:
# Function to extract questions from the given format
def extract_questions(question_text):
    try:
        # Use regex to extract text between "Option A:" and "Option B:"
        match = re.search(r"Option A:\s*(.*?)\s*\|\s*Option B:\s*(.*)", str(question_text), re.DOTALL)
        if match:
            return match.group(1).strip(), match.group(2).strip()
    except Exception as e:
        print(f"Error processing question: {e}")
    return None, None

# Ensure `question_one` and `question_two` columns exist
updated_df[["question_one", "question_two"]] = None  

# Create a mask for rows where `Item_x` contains "Personality Choice"
mask = updated_df["Item_x"].str.contains("Personality Choice", na=False, regex=True)

# Apply function to extract `question_one` and `question_two`
extracted_questions = updated_df.loc[mask, "Question"].astype(str).apply(lambda x: extract_questions(x))

# Convert extracted tuples into a DataFrame
extracted_questions_df = extracted_questions.apply(pd.Series)

In [29]:
# Assign extracted questions to the correct columns
updated_df.loc[mask, ["question_one", "question_two"]] = extracted_questions_df.rename(columns={0: "question_one", 1: "question_two"})

# Processing 
updated_df["question_one"] = updated_df["question_one"].astype(str).str.strip('.')
updated_df["question_two"] = updated_df["question_two"].astype(str).str.strip('.')

In [30]:
# Updated function to find ratings based on question_one and question_two matching Item_y within the same Job
def find_ratings(row, df):
    if pd.isna(row["question_one"]) or pd.isna(row["question_two"]):
        return np.nan, np.nan

    # Filter the dataset to only include rows with the same Job
    job_group = df[df["Job"] == row["Job"]]

    # Find ratings where question_one matches Item_y
    rating_one = job_group.loc[job_group["Item_y"] == row["question_one"], "Response"]
    rating_one_value = rating_one.values[0] if not rating_one.empty else np.nan

    # Find ratings where question_two matches Item_y
    rating_two = job_group.loc[job_group["Item_y"] == row["question_two"], "Response"]
    rating_two_value = rating_two.values[0] if not rating_two.empty else np.nan

    return float(rating_one_value), float(rating_two_value)

# Apply the function to each row to extract ratings based on Item_y matches
updated_df[["rating_one", "rating_two"]] = updated_df.apply(lambda row: find_ratings(row, updated_df), axis=1).apply(pd.Series)

In [31]:
mask = updated_df['rating_one'].notna() & updated_df['rating_two'].notna()

updated_df.loc[mask, 'Response'] = np.where(
    updated_df.loc[mask, 'rating_one'] > updated_df.loc[mask, 'rating_two'], -2,
    np.where(
        updated_df.loc[mask, 'rating_one'] < updated_df.loc[mask, 'rating_two'],  2, 
        0
    )
)

In [32]:
updated_df = updated_df[['Job', 'Question', 'Format', 'Item_x', 'Response']]
updated_df.rename(columns={'Item_x': 'Item'}, inplace=True)

# Interview

In [33]:
# Loading interview response
interview_sjt = pd.read_csv('intermediate_data/interview_sjt.csv')
interview_bq = pd.read_csv('intermediate_data/interview_bq.csv')

In [34]:
mask = updated_df["Item"].str.contains("Interview 1", case=False, na=False)

updated_df.loc[mask, "Response"] = interview_sjt['Interview'][0]
mask = updated_df["Item"].str.contains("Interview 2", case=False, na=False)

updated_df.loc[mask, "Response"] = interview_sjt['Interview'][1]
mask = updated_df["Item"].str.contains("Interview 3", case=False, na=False)

updated_df.loc[mask, "Response"] = interview_sjt['Interview'][2]

mask = updated_df["Item"].str.contains("Interview 4", case=False, na=False)

updated_df.loc[mask, "Response"] =  interview_sjt['Interview'][3]

mask = updated_df["Item"].str.contains("Interview 5", case=False, na=False)

updated_df.loc[mask, "Response"] = interview_sjt['Interview'][4]

In [35]:
def get_first_sentence(s):
    s = s.split('\n')
    return s[0]

In [36]:
interview_bq['clean_text'] = interview_bq['Interview'].apply(get_first_sentence)

In [37]:
# Merge and update Response column
updated_df = updated_df.merge(interview_bq,
                              left_on=['Job','Question'], 
                              right_on=['Role', 'Question_Set'],
                              how='left')

# Merge and update Response column
updated_df['Response'] = updated_df['clean_text'].combine_first(updated_df['Response'])

In [38]:
updated_df = updated_df.drop(columns=['Role', 'Question_Set', 'Interview', 'clean_text'])

# Output

In [39]:
# Filter rows where 'Item' contains 'Likert' and 'Response' is 1
mask = (updated_df['Item'].str.contains('Likert', na=False, case=False)) & (updated_df['Response'] == 3)

# Get indices of rows that match the condition
indices = updated_df[mask].index

# Select 20% of those indices randomly
num_to_modify = int(1 * len(indices))  # 20% of the matching rows
random_indices = np.random.choice(indices, size=num_to_modify, replace=False)

# Update the selected rows
updated_df.loc[random_indices, 'Response'] = 5

In [40]:
mask = (
    updated_df["Item"].str.contains("Choice", case=False, na=False) &
    (
        (updated_df["Response"].isna()) |                # NaN
        (updated_df["Response"].str.strip() == "")       # empty or just whitespace
    )
)

# Update the Response column for those rows
updated_df.loc[mask, "Response"] = 0

mask = (
    updated_df["Item"].str.contains("Resume", case=False, na=False) &
    (
        (updated_df["Response"].isna()) |                # NaN
        (updated_df["Response"].str.strip() == "")       # empty or just whitespace
    )
)

# Update the Response column for those rows
updated_df.loc[mask, "Response"] = "None"

In [41]:
updated_df.to_excel('result/final_result.xlsx', index=False)

In [42]:
updated_df.to_csv('result/final_result.csv', index=False)